In [9]:
!pip -q install -U geemap
!pip install -q pycrs

import os
import glob
import ee
import geemap
from google.colab import drive

drive.mount('/content/drive')

ee.Authenticate()
ee.Initialize(project='ee-moiduranperu')


  Preparing metadata (setup.py) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# lab adaptation
AOI_shp = "/content/drive/MyDrive/MOISES_GEOG6855_Lab_1/AOI_Shapefiles/AOI_NorthAL.shp"

AOI_geemap = geemap.shp_to_ee(AOI_shp)

collection = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA")
    .filterBounds(AOI_geemap)
    .filterDate('2019-05-25', '2019-05-26')
    .filter(ee.Filter.lt('CLOUD_COVER', 20))
)

img_ids = collection.aggregate_array('system:id').getInfo()
print("Available Landsat-8 images:\n", img_ids)

bands = ['B1', 'B2', 'B3', 'B4']

download_root = "/content/drive/MyDrive/MOISES_GEOG6855_Lab_1/AOI_Shapefiles/download"
os.makedirs(download_root, exist_ok=True)

# download first two images
for i in range(min(2, len(img_ids))):
    img = ee.Image(img_ids[i]).select(bands)
    tile_name = img_ids[i].split('/')[-1]
    out_name = f"{tile_name}_4band.tif"
    out_path = os.path.join(download_root, out_name)

    geemap.download_ee_image(img, out_path, crs="EPSG:4326", scale=30)
    print("Saved:", out_path)


Available Landsat-8 images:
 ['LANDSAT/LC08/C02/T1_TOA/LC08_022035_20190525', 'LANDSAT/LC08/C02/T1_TOA/LC08_022036_20190525']


/usr/local/lib/python3.12/dist-packages/geemap/common.py:12434: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


LC08_022035_20190525:   0%|          |0/320 tiles [00:00<?]

Saved: /content/drive/MyDrive/MOISES_GEOG6855_Lab_1/AOI_Shapefiles/download/LC08_022035_20190525_4band.tif


LC08_022036_20190525:   0%|          |0/320 tiles [00:00<?]

Saved: /content/drive/MyDrive/MOISES_GEOG6855_Lab_1/AOI_Shapefiles/download/LC08_022036_20190525_4band.tif


In [11]:
# Assignment 2

import os
import glob
import ee
import geemap

AOI_FOLDER = "/content/drive/MyDrive/MOISES_GEOG6855_Lab_1/AOI_Shapefiles"
ROOT_OUT = "/content/drive/MyDrive/MOISES_GEOG6855_Lab_1/Lab1_Assignment2"

# Date window
DATE_START = "2019-05-25"
DATE_END   = "2019-05-26"

# Cloud limit (<= 60% required)
CLOUD_LIMIT = 60

# Sentinel-2 collection (SR harmonized is a common, stable choice)
S2_COLLECTION_ID = "COPERNICUS/S2_SR_HARMONIZED"

# 10m + 20m bands ONLY (S2 SR)
BANDS_10M = ["B2", "B3", "B4", "B8"]  # Blue, Green, Red, NIR (10m)
BANDS_20M = ["B5", "B6", "B7", "B8A", "B11", "B12"]  # Red-edge + SWIR (20m)
BANDS = BANDS_10M + BANDS_20M

# Resample to 30m during download
DOWNLOAD_SCALE_M = 30
DOWNLOAD_CRS = "EPSG:4326"

# ---- Make sure output root exists ----
os.makedirs(ROOT_OUT, exist_ok=True)

# ---- Find all shapefiles in AOI folder (glob) ----
shp_paths = sorted(glob.glob(os.path.join(AOI_FOLDER, "*.shp")))
print("Found shapefiles:")
for p in shp_paths:
    print(" -", p)

if len(shp_paths) == 0:
    raise FileNotFoundError(f"No .shp files found in: {AOI_FOLDER}")

# ---- Helper: make a safe folder name ----
def safe_name(x: str) -> str:
    return "".join([c if c.isalnum() or c in ["_", "-", "."] else "_" for c in x])

# ---- Loop AOIs ----
for shp_path in shp_paths:
    aoi_name = safe_name(os.path.splitext(os.path.basename(shp_path))[0])
    print("\n" + "="*70)
    print(f"AOI: {aoi_name}")
    print("="*70)

    # Convert shapefile to EE object
    aoi_ee = geemap.shp_to_ee(shp_path)

    # Build Sentinel-2 collection
    s2 = (
        ee.ImageCollection(S2_COLLECTION_ID)
        .filterBounds(aoi_ee)
        .filterDate(DATE_START, DATE_END)
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", CLOUD_LIMIT))
    )

    # Get all tile IDs
    tile_ids = s2.aggregate_array("system:id").getInfo()
    print(f"Tiles found (cloud <= {CLOUD_LIMIT}%): {len(tile_ids)}")
    if len(tile_ids) == 0:
        print("No intersecting Sentinel-2 tiles for this AOI/date/cloud filter.")
        continue

    # AOI output folder
    aoi_out_dir = os.path.join(ROOT_OUT, aoi_name)
    os.makedirs(aoi_out_dir, exist_ok=True)

    # Download all tiles
    for tid in tile_ids:
        tile_name = safe_name(tid.split("/")[-1])   # folder name = tile name
        tile_dir = os.path.join(aoi_out_dir, tile_name)
        os.makedirs(tile_dir, exist_ok=True)

        out_tif = os.path.join(tile_dir, f"{tile_name}.tif")

        img = ee.Image(tid).select(BANDS).clip(aoi_ee)

        print(f"\nDownloading tile: {tile_name}")
        print(" ->", out_tif)

        # This will resample to 30m via scale=30
        geemap.download_ee_image(
            img,
            filename=out_tif,
            scale=DOWNLOAD_SCALE_M,
            crs=DOWNLOAD_CRS
        )

print("\nDONE. All downloads (if any) are in:", ROOT_OUT)


Found shapefiles:
 - /content/drive/MyDrive/MOISES_GEOG6855_Lab_1/AOI_Shapefiles/AOI_Nebe.shp
 - /content/drive/MyDrive/MOISES_GEOG6855_Lab_1/AOI_Shapefiles/AOI_NorthAL.shp
 - /content/drive/MyDrive/MOISES_GEOG6855_Lab_1/AOI_Shapefiles/AOI_Tulsa.shp

AOI: AOI_Nebe
Tiles found (cloud <= 60%): 3

 -> /content/drive/MyDrive/MOISES_GEOG6855_Lab_1/Lab1_Assignment2/AOI_Nebe/20190525T170859_20190525T171218_T14TPM/20190525T170859_20190525T171218_T14TPM.tif


20190525T170859_20190525T171218_T14TPM:   0%|          |0/80 tiles [00:00<?]


 -> /content/drive/MyDrive/MOISES_GEOG6855_Lab_1/Lab1_Assignment2/AOI_Nebe/20190525T170859_20190525T171218_T14TQM/20190525T170859_20190525T171218_T14TQM.tif


20190525T170859_20190525T171218_T14TQM:   0%|          |0/160 tiles [00:00<?]


 -> /content/drive/MyDrive/MOISES_GEOG6855_Lab_1/Lab1_Assignment2/AOI_Nebe/20190525T170859_20190525T171218_T15TTG/20190525T170859_20190525T171218_T15TTG.tif


...


AOI: AOI_NorthAL
Tiles found (cloud <= 60%): 2

 -> /content/drive/MyDrive/MOISES_GEOG6855_Lab_1/Lab1_Assignment2/AOI_NorthAL/20190525T161901_20190525T163308_T16SFC/20190525T161901_20190525T163308_T16SFC.tif


20190525T161901_20190525T163308_T16SFC:   0%|          |0/160 tiles [00:00<?]


 -> /content/drive/MyDrive/MOISES_GEOG6855_Lab_1/Lab1_Assignment2/AOI_NorthAL/20190525T161901_20190525T163308_T16SFD/20190525T161901_20190525T163308_T16SFD.tif


20190525T161901_20190525T163308_T16SFD:   0%|          |0/160 tiles [00:00<?]


AOI: AOI_Tulsa
Tiles found (cloud <= 60%): 0
No intersecting Sentinel-2 tiles for this AOI/date/cloud filter.

DONE. All downloads (if any) are in: /content/drive/MyDrive/MOISES_GEOG6855_Lab_1/Lab1_Assignment2
